<a href="https://colab.research.google.com/github/codematser69/candidate-ranking--dashboard/blob/main/panda_tech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
print("Setup working!")

Setup working!


from google.colab import drive
drive.mount('https://drive.google.com/drive/folders/1JUkYTcpEcUmSB-JaTb375WwdKvs0fcs3?usp=drive_link')



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.listdir('/content/drive/MyDrive/redrob_hackathon')

['candidates.jsonl',
 'job_description.docx',
 'sample_candidates.json',
 'submission_spec.docx',
 'validate_submission.py',
 'submission_metadata_template.yaml',
 'candidate_schema.json',
 'redrob_signals_doc.docx',
 'sample_submission.csv']

In [ ]:
import json

with open('/content/drive/MyDrive/redrob_hackathon/sample_candidates.json', 'r') as f:
    candidates = json.load(f)

print(len(candidates))     # should print 50
print(candidates[0])       # should print one full candidate record

50
{'candidate_id': 'CAND_0000001', 'profile': {'anonymized_name': 'Ira Vora', 'headline': 'Backend Engineer | SQL, Spark, Cloud', 'summary': "Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side — Python, SQL, Spark, Airflow, warehouse design — and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice.", 'location': 'Toronto', 'country': 'Canada', 'years_of_experience': 6.9, 'current_title': 'Backend Engineer', 'current_company': 'Mindtree', 'current_company_size': '10001+', 'current_industry': 'IT Services'}, 'career_hi

In [ ]:
candidates_full = []
with open('/content/drive/MyDrive/redrob_hackathon/candidates.jsonl', 'r') as f:
    for line in f:
        if line.strip():
            candidates_full.append(json.loads(line))

print(len(candidates_full))   # should print 100000

100000


In [ ]:
# ============================================================
# CELL 2: Install only what's missing on Colab
# ============================================================
!pip install -q lightgbm sentence-transformers
print("Libraries installed")

Libraries installed


In [ ]:
# ============================================================
# CELL 3: Imports & Config
# ============================================================
import json
import os
import re
import math
import time
import warnings
from datetime import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
warnings.filterwarnings("ignore")

# PATHS — adjust folder name to match YOUR Drive exactly
DRIVE_BASE = "/content/drive/MyDrive/redrob-hackathon"
CANDIDATES_PATH = f"{DRIVE_BASE}/candidates.jsonl"     # NOTE: .jsonl, not .jsonl.gz
OUTPUT_DIR = f"{DRIVE_BASE}/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Config ready")
print(f"Dataset path: {CANDIDATES_PATH}")
print(f"Output path : {OUTPUT_DIR}")


Config ready
Dataset path: /content/drive/MyDrive/redrob-hackathon/candidates.jsonl
Output path : /content/drive/MyDrive/redrob-hackathon/output


In [ ]:
# ============================================================
# CELL 4: Load all 100K candidates (plain .jsonl, no gzip needed)
# ============================================================
def load_candidates(path):
    candidates = []
    with open(path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc="Loading candidates", unit=" lines"):
            line = line.strip()
            if line:
                candidates.append(json.loads(line))
    return candidates

start = time.time()
candidates = load_candidates(CANDIDATES_PATH)
print(f"Loaded {len(candidates):,} candidates in {time.time()-start:.1f}s")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/redrob-hackathon/candidates.jsonl'